In [1]:
# Locate the repository when running from any notebook subfolder.
from pathlib import Path
import os
_notebook_start = Path.cwd().resolve()
_notebook_root = next(
    (p for p in [_notebook_start, *_notebook_start.parents]
     if (p / "docs" / "predictor_list.csv").is_file()
     and (p / "notebooks").is_dir()), None
)
if _notebook_root is None:
    raise FileNotFoundError("Cannot locate the project repository.")
os.chdir(_notebook_root)


# Data Quality and Leakage Review

## Objective

This notebook checks the quality and consistency of the official research dataset and reviews feature availability for early late-delivery prediction.

The review uses purchase time as the reference point. Features requiring information recorded after purchase are identified separately. The final prediction point must be confirmed before modeling.

The source dataset is preserved. No missing-value imputation, row removal, hypothesis testing, or model training is performed.

In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Locate the repository from either its root or notebooks directory.
current_dir = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [current_dir, *current_dir.parents]
        if (path / "data/processed/research_data.csv").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate data/processed/research_data.csv"
    )

DATA_PATH = PROJECT_ROOT / "data/processed/research_data.csv"

# Preserve identifiers as strings.
source_df = pd.read_csv(
    DATA_PATH,
    dtype={
        "order_id": "string",
        "customer_id": "string",
    },
)

# Perform review calculations on a separate working copy.
df = source_df.copy(deep=True)

structure_summary = pd.DataFrame({
    "Check": [
        "Total rows",
        "Total columns",
        "Missing order_id",
        "Duplicate order_id rows",
        "Invalid target values",
    ],
    "Value": [
        len(df),
        df.shape[1],
        df["order_id"].isna().sum(),
        df["order_id"].duplicated().sum(),
        (
            df["is_late"].notna()
            & ~df["is_late"].isin([0, 1])
        ).sum(),
    ],
})

print("Dataset:", DATA_PATH)
display(structure_summary)

Dataset: C:\Users\LENOVO\OneDrive\Desktop\PTTK\data\processed\research_data.csv


,Check,Value
0,Total rows,99441
1,Total columns,34
2,Missing order_id,0
3,Duplicate order_id rows,0
4,Invalid target values,0


## Timestamp Parsing and Missing Values

Order timestamps are converted to datetime format. Missing source values are distinguished from values that cannot be parsed.

Missing values are also counted for the six temporal features. These values are preserved without imputation.

In [3]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

date_check_rows = []

for column in date_columns:
    # Always parse from the unchanged source copy.
    original = source_df[column]

    converted = pd.to_datetime(
        original,
        format="ISO8601",
        errors="coerce",
    )

    # A nonmissing source value becoming NaT is a parsing failure.
    invalid_dates = original.notna() & converted.isna()

    date_check_rows.append({
        "Feature": column,
        "Source missing": int(original.isna().sum()),
        "Invalid dates": int(invalid_dates.sum()),
        "Missing after parsing": int(converted.isna().sum()),
    })

    df[column] = converted

date_parse_summary = pd.DataFrame(date_check_rows)

print("TIMESTAMP PARSING")
display(date_parse_summary)

assert date_parse_summary["Invalid dates"].eq(0).all(), (
    "Unparseable timestamps found. Review them before continuing."
)

temporal_features = [
    "purchase_hour",
    "purchase_is_weekend",
    "estimated_window_days",
    "approval_delay_hours",
    "actual_delivery_days",
    "delivery_delay_days",
]

temporal_missing_summary = pd.DataFrame({
    "Feature": temporal_features,
    "Missing orders": [
        int(df[column].isna().sum())
        for column in temporal_features
    ],
    "Missing percent": [
        round(df[column].isna().mean() * 100, 2)
        for column in temporal_features
    ],
})

print("TEMPORAL FEATURE MISSINGNESS")
display(temporal_missing_summary)

TIMESTAMP PARSING


,Feature,Source missing,Invalid dates,Missing after parsing
0,order_purchase_timestamp,0,0,0
1,order_approved_at,160,0,160
2,order_delivered_carrier_date,1783,0,1783
3,order_delivered_customer_date,2965,0,2965
4,order_estimated_delivery_date,0,0,0


TEMPORAL FEATURE MISSINGNESS


,Feature,Missing orders,Missing percent
0,purchase_hour,0,0.00
1,purchase_is_weekend,0,0.00
2,estimated_window_days,0,0.00
3,approval_delay_hours,160,0.16
4,actual_delivery_days,2965,2.98
5,delivery_delay_days,2965,2.98


## Temporal Feature Consistency

The six temporal features are independently recalculated from their source timestamps. Stored and recalculated values are compared, including their missing-value patterns.

Calendar-day differences use normalized dates. Approval delay and actual delivery duration use continuous elapsed time. An absolute tolerance of 1e-8 is used for numeric comparisons.

In [4]:
import numpy as np

purchase = df["order_purchase_timestamp"]
approved = df["order_approved_at"]
delivered = df["order_delivered_customer_date"]
estimated = df["order_estimated_delivery_date"]

# Recalculate features without overwriting stored columns.
recalculated_features = {
    "purchase_hour": purchase.dt.hour,

    "purchase_is_weekend": (
        purchase.dt.dayofweek.ge(5)
        .astype("Int64")
        .where(purchase.notna())
    ),

    "estimated_window_days": (
        estimated.dt.normalize() - purchase.dt.normalize()
    ).dt.days,

    "approval_delay_hours": (
        approved - purchase
    ).dt.total_seconds() / 3600,

    "actual_delivery_days": (
        delivered - purchase
    ).dt.total_seconds() / 86400,

    "delivery_delay_days": (
        delivered.dt.normalize() - estimated.dt.normalize()
    ).dt.days,
}

consistency_rows = []

for feature, expected in recalculated_features.items():
    original = source_df[feature]
    stored = pd.to_numeric(original, errors="coerce")

    invalid_numeric = original.notna() & stored.isna()

    missing_mismatch = stored.isna() != expected.isna()
    comparable = stored.notna() & expected.notna()

    values_match = np.isclose(
        stored.loc[comparable].to_numpy(dtype=float),
        expected.loc[comparable].to_numpy(dtype=float),
        rtol=0,
        atol=1e-8,
    )

    consistency_rows.append({
        "Feature": feature,
        "Invalid numeric": int(invalid_numeric.sum()),
        "Missing mismatches": int(missing_mismatch.sum()),
        "Comparable orders": int(comparable.sum()),
        "Value mismatches": int((~values_match).sum()),
    })

temporal_consistency_summary = pd.DataFrame(consistency_rows)

display(temporal_consistency_summary)

check_columns = [
    "Invalid numeric",
    "Missing mismatches",
    "Value mismatches",
]

assert temporal_consistency_summary[check_columns].eq(0).all().all(), (
    "Temporal inconsistencies found. Review before continuing."
)

print("PASS: All six temporal features match their source timestamps.")

,Feature,Invalid numeric,Missing mismatches,Comparable orders,Value mismatches
0,purchase_hour,0,0,99441,0
1,purchase_is_weekend,0,0,99441,0
2,estimated_window_days,0,0,99441,0
3,approval_delay_hours,0,0,99281,0
4,actual_delivery_days,0,0,96476,0
5,delivery_delay_days,0,0,96476,0


PASS: All six temporal features match their source timestamps.


## Target Eligibility and Consistency

The late-delivery label is independently reconstructed using the agreed eligibility rules. Eligible orders must have delivered status, valid purchase and delivery timestamps, and actual delivery not preceding purchase.

Late delivery is determined by comparing actual and estimated calendar dates. Orders without valid labels remain excluded from late-rate calculations.

In [5]:
# Identify stored valid labels.
valid_label = df["is_late"].isin([0, 1])

# Reconstruct the eligibility rule used in Stage 1.
eligible = (
    df["order_status"].eq("delivered")
    & purchase.notna()
    & delivered.notna()
    & estimated.notna()
    & delivered.ge(purchase)
)

# Reconstruct labels separately; preserve the stored target.
reconstructed_target = pd.Series(
    pd.NA,
    index=df.index,
    dtype="Int64",
)

reconstructed_target.loc[eligible] = (
    delivered.loc[eligible].dt.normalize()
    > estimated.loc[eligible].dt.normalize()
).astype("int64")

# Check eligibility and label values separately.
eligible_without_label = eligible & ~valid_label
ineligible_with_label = ~eligible & valid_label
comparable_labels = eligible & valid_label

label_value_mismatches = int(
    df.loc[comparable_labels, "is_late"]
    .ne(reconstructed_target.loc[comparable_labels])
    .sum()
)

missing_delay_with_label = (
    valid_label & df["delivery_delay_days"].isna()
)

delay_sign_mismatches = int(
    df.loc[valid_label, "is_late"]
    .ne(
        df.loc[valid_label, "delivery_delay_days"]
        .gt(0)
        .astype(int)
    )
    .sum()
)

# Investigate orders with a delay value but no valid label.
unlabeled_with_delay = (
    ~valid_label
    & df["delivery_delay_days"].notna()
)

delivered_without_label = (
    df["order_status"].eq("delivered")
    & ~valid_label
)

target_validation_summary = pd.DataFrame({
    "Check": [
        "Eligible orders",
        "Orders with valid labels",
        "Late orders",
        "Not Late orders",
        "Missing target",
        "Eligible orders without a label",
        "Ineligible orders with a label",
        "Label value mismatches",
        "Labeled orders missing delivery delay",
        "Delay sign mismatches",
        "Unlabeled orders with delivery delay",
        "Delivered orders without a label",
    ],
    "Orders": [
        int(eligible.sum()),
        int(valid_label.sum()),
        int(df["is_late"].eq(1).sum()),
        int(df["is_late"].eq(0).sum()),
        int(df["is_late"].isna().sum()),
        int(eligible_without_label.sum()),
        int(ineligible_with_label.sum()),
        label_value_mismatches,
        int(missing_delay_with_label.sum()),
        delay_sign_mismatches,
        int(unlabeled_with_delay.sum()),
        int(delivered_without_label.sum()),
    ],
})

display(target_validation_summary)

print("STATUS OF UNLABELED ORDERS WITH DELIVERY DELAY")
display(
    df.loc[unlabeled_with_delay, "order_status"]
    .value_counts(dropna=False)
    .rename_axis("Order status")
    .to_frame("Orders")
)

assert not eligible_without_label.any()
assert not ineligible_with_label.any()
assert label_value_mismatches == 0
assert not missing_delay_with_label.any()
assert delay_sign_mismatches == 0

print("PASS: Target eligibility and delivery-delay consistency checks.")

,Check,Orders
0,Eligible orders,96470
1,Orders with valid labels,96470
2,Late orders,6534
3,Not Late orders,89936
4,Missing target,2971
5,Eligible orders without a label,0
6,Ineligible orders with a label,0
7,Label value mismatches,0
8,Labeled orders missing delivery delay,0
9,Delay sign mismatches,0


STATUS OF UNLABELED ORDERS WITH DELIVERY DELAY


,Orders
Order status,
canceled,6


PASS: Target eligibility and delivery-delay consistency checks.


## Value Ranges and Timestamp Chronology

Temporal features are checked against their expected domains. Purchase hour must be an integer from 0 to 23, and the weekend indicator must be binary. Estimated delivery window, approval delay, and actual delivery duration must be nonnegative.

Negative delivery delay represents early delivery and is valid. Timestamp chronology anomalies are reported separately. Original values and orders are retained for review.

In [6]:
range_rules = {
    "Purchase hour outside 0-23 or non-integer": (
        df["purchase_hour"].notna()
        & (
            ~df["purchase_hour"].between(0, 23)
            | df["purchase_hour"].mod(1).ne(0)
        )
    ),

    "Weekend indicator outside {0, 1}": (
        df["purchase_is_weekend"].notna()
        & ~df["purchase_is_weekend"].isin([0, 1])
    ),

    "Negative estimated delivery window": (
        df["estimated_window_days"] < 0
    ),

    "Negative approval delay": (
        df["approval_delay_hours"] < 0
    ),

    "Negative actual delivery duration": (
        df["actual_delivery_days"] < 0
    ),

    "Non-integer estimated calendar-day window": (
        df["estimated_window_days"].notna()
        & df["estimated_window_days"].mod(1).ne(0)
    ),

    "Non-integer calendar-day delivery delay": (
        df["delivery_delay_days"].notna()
        & df["delivery_delay_days"].mod(1).ne(0)
    ),
}

range_summary = pd.DataFrame([
    {
        "Check": check,
        "Flagged orders": int(mask.sum()),
    }
    for check, mask in range_rules.items()
])

display(range_summary)

assert range_summary["Flagged orders"].eq(0).all(), (
    "Values outside expected domains were found."
)

print("PASS: All recorded temporal range checks.")

,Check,Flagged orders
0,Purchase hour outside 0-23 or non-integer,0
1,"Weekend indicator outside {0, 1}",0
2,Negative estimated delivery window,0
3,Negative approval delay,0
4,Negative actual delivery duration,0
5,Non-integer estimated calendar-day window,0
6,Non-integer calendar-day delivery delay,0


PASS: All recorded temporal range checks.


In [7]:
carrier = df["order_delivered_carrier_date"]

chronology_rules = [
    ("Approval before purchase", approved, purchase),
    ("Carrier handoff before purchase", carrier, purchase),
    ("Customer delivery before carrier handoff", delivered, carrier),
    ("Customer delivery before purchase", delivered, purchase),
    ("Carrier handoff before approval", carrier, approved),
]

chronology_rows = []

for check, later_event, earlier_event in chronology_rules:
    comparable = (
        later_event.notna()
        & earlier_event.notna()
    )

    abnormal = (
        comparable
        & later_event.lt(earlier_event)
    )

    chronology_rows.append({
        "Check": check,
        "Comparable orders": int(comparable.sum()),
        "Flagged orders": int(abnormal.sum()),
    })

chronology_summary = pd.DataFrame(chronology_rows)

display(chronology_summary)

print(
    "Chronology anomalies are recorded for review. "
    "No timestamps or orders have been changed."
)

,Check,Comparable orders,Flagged orders
0,Approval before purchase,99281,0
1,Carrier handoff before purchase,97658,166
2,Customer delivery before carrier handoff,96475,23
3,Customer delivery before purchase,96476,0
4,Carrier handoff before approval,97644,1359


Chronology anomalies are recorded for review. No timestamps or orders have been changed.


## Temporal Outliers

Potential outliers are identified using Tukey's 1.5 × IQR rule for four duration features. The calculation uses all nonmissing observations in the research dataset.

Values outside the IQR fences are retained. These flags describe unusual observations and do not establish that the values are incorrect.

In [8]:
duration_features = [
    "estimated_window_days",
    "approval_delay_hours",
    "actual_delivery_days",
    "delivery_delay_days",
]

outlier_rows = []

for feature in duration_features:
    values = df[feature].dropna()

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    outlier_mask = (
        values.lt(lower_fence)
        | values.gt(upper_fence)
    )

    outlier_rows.append({
        "Feature": feature,
        "Nonmissing orders": len(values),
        "Minimum": values.min(),
        "Maximum": values.max(),
        "Lower fence": lower_fence,
        "Upper fence": upper_fence,
        "Flagged orders": int(outlier_mask.sum()),
        "Flagged percent": outlier_mask.mean() * 100,
    })

outlier_summary = pd.DataFrame(outlier_rows)

# Round for display only; calculations use unrounded values.
display(outlier_summary.round(4))

long_delivery_count = int(
    df["actual_delivery_days"].gt(180).sum()
)

print(
    "Orders with actual delivery duration above 180 days:",
    long_delivery_count,
)

print("All observations are retained; no outlier treatment is applied.")

,Feature,Nonmissing orders,Minimum,Maximum,Lower fence,Upper fence,Flagged orders,Flagged percent
0,estimated_window_days,99441,2.0000,156.0000,4.0000,44.0000,2464,2.4779
1,approval_delay_hours,99281,0.0000,4509.1806,-21.3337,36.1296,9141,9.2072
2,actual_delivery_days,96476,0.5334,209.6286,-6.6645,29.1512,4899,5.0779
3,delivery_delay_days,96476,-147.0000,188.0000,-32.0000,8.0000,4301,4.4581


Orders with actual delivery duration above 180 days: 14
All observations are retained; no outlier treatment is applied.


## EDA Sample Coverage and Promised Delivery Duration

Feature availability is examined within the valid labeled sample. Missing-value counts for this sample are distinguished from counts for the full dataset.

The promised delivery duration is independently recalculated from the approval and estimated delivery calendar dates. Negative durations are treated as missing, following the existing dataset definition.

In [9]:
eda_features = [
    "purchase_weekday",
    "purchase_month",
    "customer_state",
    "distance_km_mean",
    "freight_value",
    "estimated_window_days",
    "delivery_delay_days",
    "promised_days",
    "items_value",
    "seller_count",
    "has_interstate_seller",
]

labeled_df = df.loc[valid_label].copy()

eda_sample_summary = pd.DataFrame([
    {
        "Feature": feature,
        "Labeled orders": len(labeled_df),
        "Available orders": int(labeled_df[feature].notna().sum()),
        "Missing orders": int(labeled_df[feature].isna().sum()),
        "Missing percent": (
            labeled_df[feature].isna().mean() * 100
        ),
    }
    for feature in eda_features
])

display(eda_sample_summary.round(2))

assert (
    eda_sample_summary["Available orders"]
    + eda_sample_summary["Missing orders"]
).eq(len(labeled_df)).all()

print("Valid labeled sample:", len(labeled_df))

,Feature,Labeled orders,Available orders,Missing orders,Missing percent
0,purchase_weekday,96470,96470,0,0.00
1,purchase_month,96470,96470,0,0.00
2,customer_state,96470,96470,0,0.00
3,distance_km_mean,96470,95990,480,0.50
4,freight_value,96470,96470,0,0.00
5,estimated_window_days,96470,96470,0,0.00
6,delivery_delay_days,96470,96470,0,0.00
7,promised_days,96470,96452,18,0.02
8,items_value,96470,96470,0,0.00
9,seller_count,96470,96470,0,0.00


Valid labeled sample: 96470


In [10]:
# Calendar days from approval to the estimated delivery date.
raw_promised_days = (
    estimated.dt.normalize()
    - approved.dt.normalize()
).dt.days

# Preserve the existing rule: negative durations become missing.
expected_promised_days = raw_promised_days.where(
    raw_promised_days.ge(0)
)

stored_promised_days = pd.to_numeric(
    source_df["promised_days"],
    errors="raise",
)

promised_missing_mismatches = int(
    (
        stored_promised_days.isna()
        != expected_promised_days.isna()
    ).sum()
)

comparable_promised = (
    stored_promised_days.notna()
    & expected_promised_days.notna()
)

promised_value_mismatches = int(
    (
        ~np.isclose(
            stored_promised_days.loc[
                comparable_promised
            ].to_numpy(dtype=float),
            expected_promised_days.loc[
                comparable_promised
            ].to_numpy(dtype=float),
            rtol=0,
            atol=1e-8,
        )
    ).sum()
)

reason_masks = {
    "Missing approval timestamp": approved.isna(),
    "Missing estimated delivery timestamp": estimated.isna(),
    "Negative raw promised duration": raw_promised_days.lt(0),
    "Total stored promised_days missing": stored_promised_days.isna(),
}

promised_missing_summary = pd.DataFrame([
    {
        "Reason": reason,
        "All orders": int(mask.sum()),
        "Labeled orders": int((mask & valid_label).sum()),
    }
    for reason, mask in reason_masks.items()
])

display(promised_missing_summary)

print("Missing-pattern mismatches:", promised_missing_mismatches)
print("Value mismatches:", promised_value_mismatches)

assert promised_missing_mismatches == 0
assert promised_value_mismatches == 0

print("PASS: promised_days matches the documented definition.")

,Reason,All orders,Labeled orders
0,Missing approval timestamp,160,14
1,Missing estimated delivery timestamp,0,0
2,Negative raw promised duration,10,4
3,Total stored promised_days missing,170,18


Missing-pattern mismatches: 0
Value mismatches: 0
PASS: promised_days matches the documented definition.


## Feature Availability and Data Leakage

All research columns are reviewed against purchase-time prediction. The classification follows each variable's source and information dependencies.

Conditional features require confirmation that their values were available at the prediction point. Post-purchase information is excluded from purchase-time predictors. Identifiers support traceability, and is_late remains the target.

The final prediction point must be confirmed before modeling. Approval-time prediction may permit some approval-based variables, but it does not permit future delivery outcomes.

In [11]:
leakage_rows = []

def add_review(features, availability, role, reason):
    for feature in features:
        leakage_rows.append({
            "feature": feature,
            "availability": availability,
            "modeling_role": role,
            "reason": reason,
        })


# Identifiers and target.
add_review(
    ["order_id", "customer_id"],
    "Identifier",
    "Traceability only",
    "Use for linkage and audits; exclude from default predictors. "
    "customer_id does not identify the same person across orders."
)

add_review(
    ["is_late"],
    "Target",
    "Outcome; never an input feature",
    "Constructed using final eligibility and actual delivery."
)


# Information directly available at purchase.
add_review(
    ["order_purchase_timestamp"],
    "Available at purchase",
    "Time splitting and feature engineering source",
    "Order placement timestamp."
)

add_review(
    ["purchase_weekday", "purchase_hour", "purchase_is_weekend"],
    "Available at purchase",
    "Candidate predictor",
    "Derived directly from the purchase timestamp."
)

add_review(
    ["purchase_month"],
    "Available at purchase",
    "Time splitting and EDA; predictor design requires review",
    "Year-month is known at purchase; encoding and future-month "
    "generalization require a modeling decision."
)


# Conditional information.
add_review(
    ["order_estimated_delivery_date", "estimated_window_days"],
    "Conditional",
    "Candidate source or predictor after verification",
    "Requires the original estimate available at purchase, "
    "without later revisions."
)

add_review(
    ["customer_state"],
    "Conditional",
    "Candidate predictor after verification",
    "Requires the shipping address recorded at purchase."
)

add_review(
    ["items_value", "freight_value", "item_count"],
    "Conditional",
    "Candidate predictor after verification",
    "Requires the purchased item list and quoted charges at "
    "the cutoff; later changes must not be included."
)

add_review(
    ["seller_count", "has_interstate_seller"],
    "Conditional",
    "Candidate predictor after verification",
    "Requires seller assignments and customer/seller states "
    "known at purchase."
)

add_review(
    ["distance_km_mean"],
    "Conditional",
    "Candidate predictor after verification",
    "Requires purchase-time sellers and locations, plus an "
    "outcome-independent geographic lookup available at the cutoff."
)

add_review(
    [
        "seller_count_with_geo",
        "seller_count_total",
        "has_partial_missing_geo",
        "has_all_missing_geo",
    ],
    "Conditional",
    "Coverage review; not approved as default predictors",
    "Verify purchase-time source availability, geographic lookup "
    "provenance, and redundancy before modeling."
)


# Post-purchase information.
add_review(
    ["order_status"],
    "Post-purchase",
    "Offline sample definition; exclude from predictors",
    "The historical status reflects later order progression, "
    "not a purchase-time status snapshot."
)

add_review(
    ["order_approved_at", "approval_delay_hours", "promised_days"],
    "Post-purchase",
    "Exclude from purchase-time predictors",
    "Requires realized approval. Reconsider only for an approved "
    "approval-time cutoff; promised_days also requires the estimate."
)

add_review(
    ["order_delivered_carrier_date"],
    "Post-purchase",
    "Retrospective EDA only",
    "Realized carrier handoff is not available by default "
    "at the early prediction point."
)

add_review(
    ["order_delivered_customer_date", "actual_delivery_days"],
    "Post-purchase",
    "Retrospective EDA only",
    "Requires actual customer delivery."
)

add_review(
    [
        "flag_approval_before_purchase",
        "flag_carrier_before_purchase",
        "flag_delivery_before_carrier",
        "flag_delivery_before_purchase",
        "flag_delivered_missing_date",
    ],
    "Post-purchase",
    "Quality review only; exclude from predictors",
    "Uses realized approval, carrier, delivery, or final-status "
    "information. Missingness may also reveal later progression."
)

add_review(
    ["delivery_delay_days"],
    "Post-purchase",
    "EDA and label validation only",
    "Direct leakage: its sign determines is_late "
    "for eligible labeled orders."
)


feature_leakage_review = pd.DataFrame(leakage_rows)

# Verify exhaustive coverage without duplicate classifications.
assert feature_leakage_review["feature"].is_unique
assert set(feature_leakage_review["feature"]) == set(df.columns)

# Follow the original dataset column order.
feature_leakage_review = (
    feature_leakage_review
    .set_index("feature")
    .reindex(list(df.columns))
    .rename_axis("feature")
    .reset_index()
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", 100,
):
    display(feature_leakage_review)

availability_summary = (
    feature_leakage_review["availability"]
    .value_counts()
    .rename_axis("Availability")
    .to_frame("Columns")
)

display(availability_summary)

print(
    f"PASS: All {len(feature_leakage_review)} columns "
    "have one availability classification."
)

,feature,availability,modeling_role,reason
0,order_id,Identifier,Traceability only,Use for linkage and audits; exclude from default predictors. customer_id does not identify the s...
1,customer_id,Identifier,Traceability only,Use for linkage and audits; exclude from default predictors. customer_id does not identify the s...
2,order_status,Post-purchase,Offline sample definition; exclude from predictors,"The historical status reflects later order progression, not a purchase-time status snapshot."
3,order_purchase_timestamp,Available at purchase,Time splitting and feature engineering source,Order placement timestamp.
4,order_approved_at,Post-purchase,Exclude from purchase-time predictors,Requires realized approval. Reconsider only for an approved approval-time cutoff; promised_days ...
5,order_delivered_carrier_date,Post-purchase,Retrospective EDA only,Realized carrier handoff is not available by default at the early prediction point.
6,order_delivered_customer_date,Post-purchase,Retrospective EDA only,Requires actual customer delivery.
7,order_estimated_delivery_date,Conditional,Candidate source or predictor after verification,"Requires the original estimate available at purchase, without later revisions."
8,flag_approval_before_purchase,Post-purchase,Quality review only; exclude from predictors,"Uses realized approval, carrier, delivery, or final-status information. Missingness may also rev..."
9,flag_carrier_before_purchase,Post-purchase,Quality review only; exclude from predictors,"Uses realized approval, carrier, delivery, or final-status information. Missingness may also rev..."


,Columns
Availability,
Post-purchase,13
Conditional,13
Available at purchase,5
Identifier,2
Target,1


PASS: All 34 columns have one availability classification.


## Source Preservation and Review Outputs

The source CSV is read again and compared with the original in-memory copy. Review calculations must not change the source dataset.

Validation tables are exported to outputs/tables/eda/data_quality for reproducibility and reporting.

In [12]:
# Confirm that the source CSV still matches the original loaded data.
source_check = pd.read_csv(
    DATA_PATH,
    dtype={
        "order_id": "string",
        "customer_id": "string",
    },
)

pd.testing.assert_frame_equal(
    source_df,
    source_check,
)

assert source_df.shape == (99441, 34)
assert source_df["order_id"].notna().all()
assert source_df["order_id"].is_unique
assert source_df["is_late"].dropna().isin([0, 1]).all()

print("PASS: Source CSV matches the original loaded data.")

# Full-dataset missingness, distinct from labeled-sample coverage.
full_missing_summary = pd.DataFrame({
    "Feature": source_df.columns.tolist(),
    "Missing orders": source_df.isna().sum().to_numpy(),
    "Missing percent": (
        source_df.isna().mean().mul(100).round(2).to_numpy()
    ),
})

print("Full-dataset missingness prepared for all 34 columns.")

PASS: Source CSV matches the original loaded data.
Full-dataset missingness prepared for all 34 columns.


In [13]:
TABLE_DIR = (
    PROJECT_ROOT / 'outputs/tables/eda/data_quality'
)

review_tables = {
    "structure_checks.csv": structure_summary,
    "date_parse_checks.csv": date_parse_summary,
    "missing_checks.csv": full_missing_summary,
    "temporal_consistency_checks.csv": temporal_consistency_summary,
    "target_checks.csv": target_validation_summary,
    "range_checks.csv": range_summary,
    "chronology_checks.csv": chronology_summary,
    "temporal_outlier_checks.csv": outlier_summary,
    "eda_sample_checks.csv": eda_sample_summary,
    "promised_missing_checks.csv": promised_missing_summary,
    "feature_leakage_review.csv": feature_leakage_review,
}

TABLE_DIR.mkdir(parents=True, exist_ok=True)

export_rows = []

for filename, table in review_tables.items():
    output_path = TABLE_DIR / filename

    table.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )

    # Check that each exported CSV can be read back.
    reloaded_table = pd.read_csv(output_path)

    assert reloaded_table.shape == table.shape, filename
    assert reloaded_table.columns.tolist() == table.columns.tolist(), (
        filename
    )

    export_rows.append({
        "File": filename,
        "Rows": len(reloaded_table),
        "Columns": len(reloaded_table.columns),
    })

export_summary = pd.DataFrame(export_rows)

display(export_summary)

print(
    f"PASS: {len(export_summary)} review tables "
    "exported and reloaded successfully."
)

,File,Rows,Columns
0,structure_checks.csv,5,2
1,date_parse_checks.csv,5,4
2,missing_checks.csv,34,3
3,temporal_consistency_checks.csv,6,5
4,target_checks.csv,12,2
5,range_checks.csv,7,2
6,chronology_checks.csv,5,3
7,temporal_outlier_checks.csv,4,8
8,eda_sample_checks.csv,11,5
9,promised_missing_checks.csv,4,3


PASS: 11 review tables exported and reloaded successfully.


## Review Conclusion

The dataset contains 99,441 orders and 34 columns. The six temporal features match their source timestamps, with no value or missing-pattern mismatches. Target reconstruction also matches all 96,470 valid labels.

Timestamp anomalies and IQR outliers remain in the dataset and are documented for further review. Missing feature values are reported using the appropriate full-dataset or labeled-sample denominator.

All 34 columns have been classified by purchase-time availability. Delivery outcomes, final status, and post-purchase quality flags must be excluded from purchase-time predictors. Conditional features require confirmation of their availability at the selected prediction point.

Before modeling, the team must confirm the prediction cutoff and feature list. Learned preprocessing must be fitted on training data only. Chronological evaluation must also account for whether training labels would have been observable at the training cutoff.

The source CSV matches the original loaded data. Detailed review results are saved in outputs/tables/eda/data_quality.